In [37]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [38]:
seed = 0
df = sns.load_dataset('tips')

In [39]:
def ols(X, y):
  return np.linalg.inv(X.T @ X) @ X.T @ y

def mse(y, pred):
  return np.mean((y - pred) ** 2)

def rmse(y, pred):
  return np.sqrt(mse(y, pred))

def mae(y, pred):
  return np.mean(np.abs(y - pred))

def r_squared(y, pred):
  mean_y = np.mean(y)
  return 1 - (np.sum((y - pred) ** 2) / np.sum((y - mean_y) ** 2))

def adjusted_r_squared(y, pred, n_features):
  return 1 - (
    ((1 - r_squared(y, pred)) * (y.shape[0] - 1)) / 
    (y.shape[0] - n_features - 1)
  )


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df):
  X = df[['total_bill']]
  y = df['tip'].to_numpy()

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_test_scaled.shape == X_test.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_test_scaled = np.concat((np.ones(X_test_scaled.shape[0]).reshape(-1, 1), X_test_scaled), axis=1)

  return X_train_standard, X_test_scaled, y_train, y_test


In [41]:
X_train, X_test, y_train, y_test = produce_training_data(df)

weights = ols(X_train, y_train)
pred_ols = X_test @ weights

pd.DataFrame([{
  "MSE": mse(y_test, pred_ols),
  "RMSE": rmse(y_test, pred_ols), 
  "MAE": mae(y_test, pred_ols), 
  "R2": r_squared(y_test, pred_ols), 
  "Adjusted_R2": adjusted_r_squared(y_test, pred_ols, X_test.shape[1] - 1),
}])

,MSE,RMSE,MAE,R2,Adjusted_R2
0,0.821309,0.906261,0.656407,0.59069,0.581981
